# Adjacent-band evaluation: raw, original adapter, clean-pair adapter

Select a **GPU runtime**, then **Runtime → Run all**. This compares three frozen costs on the same **20 new starts**, with x in [40,54] and y in [22,30]. The top edge touches the training band's lower edge at y=30. The goal remains (50,42).

There are **60 evaluation episodes** total. Both adapters are loaded from Drive. There is no fitting or experience collection; normal evaluation still encodes live camera observations. Results are saved in `adjacent_band_v1`, alongside the two source experiments.


In [ ]:
from google.colab import drive
from pathlib import Path
import json, shutil, signal, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DRIVE = Path('/content/drive/MyDrive')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_REF = 'curriculum-cost-adapter'
SOURCE_COMMIT = 'bb9bbc92483c21ef8bf1f1f54b545aebc0ed0154'
REPO = Path('/content/i-jepa-wm-adjacent')
DINO_ROOT = Path('/content/dino_wm')
PARENT = DRIVE / 'datasets/curriculum_cost_adapter/pilot_v1'
CLEAN = DRIVE / 'datasets/curriculum_cost_adapter/clean_pair_v1'
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/adjacent_band_v1'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE = DRIVE / 'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS = Path('/content/curriculum-adjacent-artifacts')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'

def shell(command, *, cwd=None, log=None):
    command = [str(arg) for arg in command]
    print('$', ' '.join(command), flush=True)
    stream = open(log, 'a', buffering=1) if log else None
    try:
        with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    if stream: stream.write(line)
                returncode = process.wait()
            except BaseException:
                process.send_signal(signal.SIGINT)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: process.terminate()
                raise
        if returncode: raise subprocess.CalledProcessError(returncode, command)
    finally:
        if stream: stream.close()


## Set up the evaluation source and frozen models

The source revision, checkpoint hashes and original adapter training runtimes are recorded. All conditions share the saved clean-training goal tokens. Reruns validate these identities and skip completed cases. Keep `SOURCE_COMMIT` at the printed revision if the branch later changes.


In [ ]:
for source in (PARENT, CLEAN):
    assert source.is_dir(), f'Missing completed experiment: {source}'
    assert OUTPUT.resolve() != source.resolve() and not OUTPUT.resolve().is_relative_to(source.resolve()), 'Output must be outside both sources'
assert (CLEAN / 'run_0/goal_tokens.pt').is_file(), 'Run the clean-pair notebook first'
if not REPO.exists():
    shell(['git', 'clone', '--branch', SOURCE_REF, '--single-branch', REPO_URL, REPO])
else:
    assert (REPO / '.git').exists(), f'{REPO} is not a Git checkout'
    assert subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Source clone has local changes'
    shell(['git', '-C', REPO, 'fetch', 'origin', f'refs/heads/{SOURCE_REF}:refs/remotes/origin/{SOURCE_REF}'])
shell(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT or f'origin/{SOURCE_REF}'])
source_commit = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print('Adjacent evaluation source revision:', source_commit)

shell([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2', 'omegaconf', 'pyyaml', 'scipy', 'einops', 'matplotlib', 'imageio', 'imageio-ffmpeg', 'pillow'])
if not DINO_ROOT.exists(): shell(['git', 'clone', 'https://github.com/gaoyuezhou/dino_wm.git', DINO_ROOT])
assert not subprocess.check_output(['git', '-C', str(DINO_ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'DINO-WM source has local changes'
shell(['git', '-C', DINO_ROOT, 'checkout', DINO_COMMIT])
ARTIFACTS.mkdir(parents=True, exist_ok=True)
ENCODER, CHECKPOINT = ARTIFACTS / 'encoder.pth.tar', ARTIFACTS / 'fixed-best.pt'
for source, destination in ((ENCODER_DRIVE, ENCODER), (WM_DRIVE, CHECKPOINT)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not destination.exists() or destination.stat().st_size != source.stat().st_size: shutil.copy2(source, destination)
COMMON = ['--parent-dir', str(PARENT), '--clean-dir', str(CLEAN), '--output-dir', str(OUTPUT),
          '--world-model-checkpoint', str(CHECKPOINT), '--encoder-checkpoint', str(ENCODER),
          '--dino-wm-root', str(DINO_ROOT), '--device', 'cuda', '--precision', 'fp32', '--encoder-batch-size', '8']
def stage(name, condition=None):
    extra = ['--condition', condition] if condition else []
    started = time.monotonic()
    shell([sys.executable, '-m', 'src.mylonas_ijepa_wm.curriculum_adjacent_band', name, *COMMON, *extra],
          cwd=REPO)
    print(f"{name} {condition or ''} took {(time.monotonic() - started) / 60:.1f} minutes")
print('GPU:', torch.cuda.get_device_name())
print('Torch:', torch.__version__)
print('Parent:', PARENT)
print('Clean adapter source:', CLEAN)
print('New output:', OUTPUT)


## Freeze the 20 paired evaluation cases

This validates both adapter sources and prepares a deterministic manifest. These same starts, goals and CEM seeds are used in every condition. The prior near and far results are preserved in their source directories.


In [ ]:
stage('prepare')
manifest = json.loads((OUTPUT / 'manifest.json').read_text())
print('Bands:', manifest['bands'])
print('Evaluation cases:', len(manifest['evaluation']))
for case in manifest['evaluation']:
    print(case['case_id'], 'start=', case['start'], 'CEM seed=', case['seed'])


## Evaluate all three costs

The raw condition runs first, followed by the original-pair and clean-pair adapters. Each has 20 cases and an independent journal. Interrupted cases are retried; their already executed actions remain in the journal. Expect evaluation to dominate runtime.

At least **18/20 raw successes** establishes competence for this screen. Both adapters are still measured if raw falls below this level, but the final report marks their retention screens as uninterpretable.


In [ ]:
for condition in ('raw', 'original', 'clean'):
    print('\nEVALUATING:', condition)
    stage('evaluate', condition)


## Read the comparison

A paired loss is a case raw solved but the adapter failed. A win is the reverse. At most two paired losses counts as retaining competence, provided raw passed its screen. These thresholds are operational choices, not significance tests.

If raw reaches 20/20, success has no room to improve: compare actions, final distance and trajectories as well. Adjacent starts are held out, but near-training trajectories may already have visited some of this band. The figure shows each case separately: start cross, goal star and 4.5-unit success circle; gray adjacent band, blue near band. Path segments connect macro observations.


In [ ]:
stage('report')
report = json.loads((OUTPUT / 'report.json').read_text())
print('ADJACENT BAND: 40 <= x <= 54; 22 <= y <= 30')
print(f"{'Condition':<22} {'Success':>8} {'Final distance':>16} {'Actions':>10}")
for condition, label in [('raw', 'Raw latent MSE'), ('original', 'Original adapter'), ('clean', 'Clean-pair adapter')]:
    row = report['conditions'][condition]
    print(f"{label:<22} {row['successes']:>5}/{row['episodes']:<2} {row['mean_final_distance']:>16.3f} {row['primitive_steps']:>10}")
    print('  Failed cases:', row['failed_case_ids'])
    print('  Interaction journal:', json.dumps(row['journal']))
print('\nRAW COMPETENCE:', json.dumps(report['raw_competence']))
print('\nPAIRED COMPARISONS WITH RAW')
print(json.dumps(report['comparisons'], indent=2))
print('\nINDIVIDUAL CASES')
for row in report['per_case']:
    print(row['case']['case_id'], 'start=', row['case']['start'], json.dumps(row['results']))
print('\nInterpretation:', report['note'])
from IPython.display import display, Image
for name in ('success_comparison.png', 'trajectories.png'):
    display(Image(filename=str(OUTPUT / name)))
print('Full report:', OUTPUT / 'report.json')


## Send back the executed notebook

Choose **File → Download → Download .ipynb** and send that file back for interpretation. All case records, the fixed manifest, journals, report and figures remain in `MyDrive/datasets/curriculum_cost_adapter/adjacent_band_v1`.
